# Getting started with GenomEn

This is a short tutorial to get started with the [genomen](https://pypi.org/project/genomen/) Python package. Here, we briefly explain how to set-up GenomEn to train models on yout own data. For more in-depth information on GenomEn please refer to the [paper]().

## Install with pip

Genomen is available on PyPI and can be installed directly without cloning the repository. This is the recommended approach for users who just want to use the package.

In [ ]:
pip install genomen

Genomen provides optional dependency groups for extended functionality. Depending on your package manager, the installation method differs. The signature to install a dependency group `dep-group` is:

In [ ]:
pip install genomen[dep-group]

GenomEn supports dependency groups `dev`, `gpu`, and `dnn`. Details on each dependency group can be found in the following install from source section. 

## Install from source

For users that want to contribute to GenomEn or adapt the code base for their own purposes, we recommend to install the package from [source](https://github.com/AI-sandbox/MetaPRS/tree/1bedf812b90137bf2935d5a487d44f9780f6d738).

### Download uv (if needed)

We decide to use [uv](https://docs.astral.sh/uv/) for dependency management. If you do not have uv installed, you will have to do so before getting started. The following command installs uv:

In [ ]:
curl -LsSf https://astral.sh/uv/install.sh | sh

### Install dependencies

Next, you will have to install the project dependencies. You can find an overview of the dependencies in the [pyproject.toml](https://github.com/AI-sandbox/MetaPRS/blob/main/pyproject.toml) file. We differentiate between 4 different groups of dependencies, each for specific use cases.

### Default dependencies

In [ ]:
uv sync

This group includes the default dependencies that have to be installed in any case. These include basic framework like numpy and pandas, and base estimator dependencies such as LGBM and XGBoost.

### Dev dependencies

This group includes dependencies that are not necessarily needed for usage of the package but will be essential for contributors actively working on the project as it includes additional dependencies such as black and pytest required for new PRs.

In [ ]:
uv sync --group dev

### GPU dependencies

As the name suggests, this dependency group is needed to run genomen on GPU. It adds GPU-aware implementations of many methods used by GenomEn such as [cuML](https://docs.rapids.ai/api/cuml/stable/). Note, that this only works for NVIDIA GPU and more specifically GPUs running on CUDA version 12. If your GPU device uses a different verion, please fork the repository and adapt the requirements or do a feature request.

In [ ]:
uv sync --group gpu

It is alway possible to downlaod multiple dependency groups simultaneously or to download them all at the same time via

In [ ]:
uv sync --all-groups

## Setting up the environment

Genomen uses environment variables loaded from a `.env` file (via `python-dotenv`). The repo already contains a `.env.template` file with the following variables

```bash
# plink 1 files
FAM_PATH=""
BED_PATH=""
BIM_PATH=""

# phenotype file
MASTER_PATH=""

# result path
RESULT_PATH="./genomen_artifacts"

# wandb
WANDB_PROJECT="GenomEn"
```

- The first block of variables configures the data source from which genotype will be loaded. For now, GenomEn only supports the Plink 1 input data format (.fam, .bed, .bim). If your data has a different format, please use external tools such as [plink2](https://www.cog-genomics.org/plink/2.0/). 
- The `MASTER_PATH` should point to the file containing phenotype and covariate data. As for now, the file must be a tab-delimited text file containing phenotype information for each sample (identified by `IID`).
- The result path is pre-configured to `./genomen_artifacts` but can be changed if needed. Model artifacts like generated annotation files and model checkpoints can be found in that location.
- Finally, if wanted, users can specify the name of the Weights & Biases project they would like training metric to be logged to. Users will be asked to log into wandb upon the first run.  

Use the following command to create your own `.env` file and subsequently update the respective fields:

In [ ]:
cp .env.template .env

Genomen uses a YAML-based configuration system that provides a declarative way to define all aspects of your experiment. Again, we provide a [template config](https://github.com/AI-sandbox/MetaPRS/blob/main/config.yml.template) with default values:

```bash
---
DataSetConfig:
  phenotype_id: "HC337"             # must be in master file
  classification: true
  file_format: "plink"              # only supports plink for now
  populations: ["white_british"]
  include_x_chromosome: false
  maf_threshold: 0.05
  sex: null
  covar_config:
    include_covars: false
    covar_keys: ["age", "sex", "Global_PC1", "Global_PC2", "Global_PC3", "Global_PC4", "Global_PC5", "Global_PC6", "Global_PC7", "Global_PC8", "Global_PC9", "Global_PC10"]
  sample_sampling:
    strat: "stratify"                 # options: "random", "stratify", "balanced"
    max_samples: null
    balance_pops: false
  variant_sampling:
    strat: "random"
    max_features: 50_000
---
GenomenModelConfig:
  covar_config:
    covar_strat: "residualization"  # options: "residualization", "predictive"
    model_config:
      model_name: linear
      hyperparameters: {}
      balance_classes: false
  geno_config:
    n_estimators: 2
    compute_interactions: True
    preprocessing_config:
      z_score_thresh: 3.0
      standard_labels: false
      feature_selection:
        method: "none"              # options: "none", "k_best", "percentile", "variance_threshold", "mutual_info", "rfe"
        k: 15_000
        percentile: 0.75
        variance_threshold: 0.05
        score_func: "f_classif"     # options: "f_classif", "f_regression", "chi2"
    model_config:
      model_name: lightgbm
      ensemble_estimator_names: []
      hyperparameters: {}
      balance_classes: true
    aggregator_config:
      filter_strat: "geq-average"   # options: "none", "positive", "geq-average", "top-p-percentile"
      agg_stat: "rank-mean"         # options: "mean", "rank-mean", "loss-weighted-average"
      p: 0.75
      temp: 0.05
---
TrainConfig:
  batch_size: 2
  n_jobs: 32
  backend: "cpu"                  # options: "cpu", "gpu"
  ram_mb: 16000
  scorer: "rocauc"                # options: "r2", "rocauc", "pearson_corr"
  patience: 30
  seed: 42
  log_with_wandb: false
  save_annotation: false
  save_model: false
  compute_shap: false
```

We will explain what each of these hyperparameters does in the [train a model notebook](https://genomen-website.vercel.app/docs/api/training).